In [16]:
%run "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_01_Bronze/Jupyter Notebooks/00_bronze_data_connector.ipynb"


Preconnectors ran successfully and connected to 'clientdatastorage' storage accounts


 'policy_df, claim_df, member_df, provider_df' are ready to use from container 'raw data'(meridian architecture of bronze) 


In [17]:
import sys
import importlib

# Add the Silver layer directory to sys.path BEFORE importing
silver_dir = "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_02_Silver"
if silver_dir not in sys.path:
    sys.path.insert(0, silver_dir)

# Now import (and reload to pick up edits)
import utils_silver
from utils_silver import *

importlib.reload(utils_silver)

paths_bronze, paths_silver, paths_gold = utils_silver.build_paths(storage_account1)
print(sorted(paths_silver.keys()))
print("Imported utils_silver from", utils_silver.__file__)

✅ utils_silver.py loaded
['_metrics', '_quarantine', '_ref_dim_channel', '_ref_dim_product_line', '_reference', 'claims', 'members', 'policies', 'providers']
Imported utils_silver from /Users/manojrammopati/Public/Projects/bupa_insurance_project/_02_Silver/utils_silver.py


In [18]:
# Cell 0b — MLflow setup
from pathlib import Path
import mlflow
import mlflow.spark

PROJECT_ROOT = Path("/Users/manojrammopati/Public/Projects/bupa_insurance_project")

# File-based tracking store inside the repo
TRACKING_URI = f"file:{PROJECT_ROOT / 'mlruns'}"
mlflow.set_tracking_uri(TRACKING_URI)

# One experiment per use case
mlflow.set_experiment("bupa_fraud_claim")

print("MLflow tracking URI:", mlflow.get_tracking_uri())


MLflow tracking URI: file:/Users/manojrammopati/Public/Projects/bupa_insurance_project/mlruns


# Cell 1 – Intro markdown

ML – Claims Risk Model (Fraud / High Cost)

**Goal:** Train and evaluate Spark ML models to predict *claims risk* using the curated
 Gold feature table `ft_claims_risk_split`.

We focus on:
- Label: `Is_Fraudulent_Claim` (binary 0/1)
- Models: Random Forest, Logistic Regression, Gradient Boosted Trees

 This notebook mirrors the structure of the policy churn ML notebook:
 1. Load Gold feature table
 2. Clean + split (train / test)
 3. Build Spark ML pipeline (indexing, OHE, assembler)
 4. Train several models and compare metrics
 5. Persist the best model to ADLS
#6. Score sample claims to show how it would work in production


# Cell 2 – Imports & configuration

In [19]:
# Cell 2 — Imports & config

from pyspark.sql import functions as F
from pyspark.sql import DataFrame

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import (
    RandomForestClassifier,
    LogisticRegression,
    GBTClassifier
)
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array

# Storage / paths (adjust only if your account or container names differ)
ACCOUNT       = "clientdatastorage"
GOLD_CONTAINER = "golddata"

FT_CLAIMS_RISK_SPLIT_PATH = (
    f"abfss://{GOLD_CONTAINER}@{ACCOUNT}.dfs.core.windows.net/ft_claims_risk_split"
)

MODEL_BASE_PATH = (
    f"abfss://{GOLD_CONTAINER}@{ACCOUNT}.dfs.core.windows.net/models"
)
MODEL_NAME      = "claims_risk_best"
MODEL_PATH      = f"{MODEL_BASE_PATH}/{MODEL_NAME}"

print("Feature table path:", FT_CLAIMS_RISK_SPLIT_PATH)
print("Model path        :", MODEL_PATH)


Feature table path: abfss://golddata@clientdatastorage.dfs.core.windows.net/ft_claims_risk_split
Model path        : abfss://golddata@clientdatastorage.dfs.core.windows.net/models/claims_risk_best


# Cell 3 – Load feature table & basic checks

In [20]:
# Cell 3 — Load ft_claims_risk_split and inspect

claims_all = (
    spark.read
         .format("delta")
         .load(FT_CLAIMS_RISK_SPLIT_PATH)
)

print("Total rows in ft_claims_risk_split:", claims_all.count())
claims_all.printSchema()
claims_all.show(5, truncate=False)


Total rows in ft_claims_risk_split: 558211
root
 |-- Claim_ID: string (nullable = true)
 |-- Member_Key: string (nullable = true)
 |-- Provider_ID: string (nullable = true)
 |-- Claim_Type_Name: string (nullable = true)
 |-- Claim_Type_Code: string (nullable = true)
 |-- Claim_Status: string (nullable = true)
 |-- Claim_Amount_GBP: double (nullable = true)
 |-- Payout_GBP: double (nullable = true)
 |-- Payout_to_Amount_Ratio: double (nullable = true)
 |-- Days_To_Settle: integer (nullable = true)
 |-- High_Cost_Claim_Flag: integer (nullable = true)
 |-- Claim_Fraud_Label: integer (nullable = true)
 |-- Provider_Fraud_Label: integer (nullable = true)
 |-- Provider_Risk_Tier: string (nullable = true)
 |-- dq_money_valid: integer (nullable = true)
 |-- dq_date_valid: integer (nullable = true)
 |-- Is_Fraudulent_Claim: integer (nullable = true)
 |-- Is_High_Cost: integer (nullable = true)
 |-- dataset_split: string (nullable = true)



+---------+----------+-----------+---------------+---------------+------------+------------------+----------+----------------------+--------------+--------------------+-----------------+--------------------+------------------+--------------+-------------+-------------------+------------+-------------+
|Claim_ID |Member_Key|Provider_ID|Claim_Type_Name|Claim_Type_Code|Claim_Status|Claim_Amount_GBP  |Payout_GBP|Payout_to_Amount_Ratio|Days_To_Settle|High_Cost_Claim_Flag|Claim_Fraud_Label|Provider_Fraud_Label|Provider_Risk_Tier|dq_money_valid|dq_date_valid|Is_Fraudulent_Claim|Is_High_Cost|dataset_split|
+---------+----------+-----------+---------------+---------------+------------+------------------+----------+----------------------+--------------+--------------------+-----------------+--------------------+------------------+--------------+-------------+-------------------+------------+-------------+
|CLM110013|BENE15435 |PRV51790   |Travel         |TRAVEL         |Settled     |244.07511199

# Cell 4 – Filter label + train/test split

We’ll model fraud risk first: Is_Fraudulent_Claim as label.

In [21]:
# Cell 4 — Filter label != null and create train/test splits

LABEL_COL = "Is_Fraudulent_Claim"

if LABEL_COL not in claims_all.columns:
    raise ValueError(f"Expected label column '{LABEL_COL}' not found in ft_claims_risk_split")

# Only keep rows where label is 0 or 1
claims_labeled = (
    claims_all
      .filter(F.col(LABEL_COL).isNotNull())
      .withColumn(LABEL_COL, F.col(LABEL_COL).cast("double"))
)

# Optional: filter to dq-valid rows only, if you want
claims_labeled = claims_labeled.filter(
    (F.col("dq_money_valid") == 1) & (F.col("dq_date_valid") == 1)
)

print("Rows with non-null label and dq-valid:", claims_labeled.count())

# Train / test split from pre-assigned column
train_df = claims_labeled.filter(F.col("dataset_split") == "train")
test_df  = claims_labeled.filter(F.col("dataset_split") == "test")

print("Train rows:", train_df.count())
print("Test rows :", test_df.count())


Rows with non-null label and dq-valid: 558211
Train rows: 446102
Test rows : 112109


# Cell 5 – Define feature columns & preprocessing function

In [22]:
# Cell 5 — Define feature columns & preprocessing helpers

# Candidate numeric features (we'll only keep those that exist)
numeric_candidates = [
    "Claim_Amount_GBP",
    "Payout_GBP",
    "Payout_to_Amount_Ratio",
    "Days_To_Settle",
]

# Candidate categorical features
categorical_candidates = [
    "Claim_Type_Name",
    "Claim_Status",
    "Claim_Type_Code",
    "Provider_Risk_Tier",
    "Provider_ID",      # optional: treat provider_id as category
]

cols = claims_labeled.columns

numeric_cols = [c for c in numeric_candidates if c in cols]
cat_cols     = [c for c in categorical_candidates if c in cols]

print("Numeric features   :", numeric_cols)
print("Categorical features:", cat_cols)

# ID / context columns to keep for interpretability (not part of features vec)
id_cols = [c for c in ["Claim_ID", "Member_Key", "Provider_ID"] if c in cols]


def prep_nulls(df: DataFrame) -> DataFrame:
    """
    Simple null-handling: numeric -> 0, categorical -> 'Unknown'.
    """
    d = df
    for c in numeric_cols:
        d = d.withColumn(c, F.when(F.col(c).isNull(), F.lit(0.0)).otherwise(F.col(c)))
    for c in cat_cols:
        d = d.withColumn(c, F.when(F.col(c).isNull(), F.lit("Unknown")).otherwise(F.col(c)))
    return d


Numeric features   : ['Claim_Amount_GBP', 'Payout_GBP', 'Payout_to_Amount_Ratio', 'Days_To_Settle']
Categorical features: ['Claim_Type_Name', 'Claim_Status', 'Claim_Type_Code', 'Provider_Risk_Tier', 'Provider_ID']


# Cell 6 – Build the Spark ML pipeline skeleton

In [23]:
# Cell 6 — Build pipeline: indexers, encoders, assembler

# 1) StringIndexers for categoricals
indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=f"{c}_idx",
        handleInvalid="keep"
    )
    for c in cat_cols
]

# 2) OneHotEncoders
encoders = [
    OneHotEncoder(
        inputCol=f"{c}_idx",
        outputCol=f"{c}_oh",
        handleInvalid="keep"
    )
    for c in cat_cols
]

# 3) VectorAssembler for final features
feature_cols = numeric_cols + [f"{c}_oh" for c in cat_cols]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="keep",
)

# Note: classifier itself will be plugged in later (RF, LR, GBT).
print("Feature columns in assembler:", feature_cols)


Feature columns in assembler: ['Claim_Amount_GBP', 'Payout_GBP', 'Payout_to_Amount_Ratio', 'Days_To_Settle', 'Claim_Type_Name_oh', 'Claim_Status_oh', 'Claim_Type_Code_oh', 'Provider_Risk_Tier_oh', 'Provider_ID_oh']


# Cell 7 – Train & evaluate helper function

In [24]:
# Cell 7 — Train & evaluate helper

def train_and_eval(pipeline: Pipeline, train_df: DataFrame, test_df: DataFrame, model_name: str):
    print(f"\n===== Training {model_name} =====")
    model = pipeline.fit(train_df)

    print(f"Scoring test set with {model_name} ...")
    pred = model.transform(test_df)

    evaluator_roc = BinaryClassificationEvaluator(
        labelCol=LABEL_COL,
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    )
    evaluator_pr = BinaryClassificationEvaluator(
        labelCol=LABEL_COL,
        rawPredictionCol="rawPrediction",
        metricName="areaUnderPR"
    )

    roc_auc = evaluator_roc.evaluate(pred)
    pr_auc  = evaluator_pr.evaluate(pred)

    # Confusion-matrix style metrics
    preds = (
        pred.select(
            LABEL_COL,
            F.col("prediction").cast("double").alias("prediction")
        )
    )

    tp = preds.filter((F.col(LABEL_COL) == 1) & (F.col("prediction") == 1)).count()
    tn = preds.filter((F.col(LABEL_COL) == 0) & (F.col("prediction") == 0)).count()
    fp = preds.filter((F.col(LABEL_COL) == 0) & (F.col("prediction") == 1)).count()
    fn = preds.filter((F.col(LABEL_COL) == 1) & (F.col("prediction") == 0)).count()

    total = tp + tn + fp + fn
    accuracy  = (tp + tn) / total if total else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall    = tp / (tp + fn) if (tp + fn) else 0.0
    f1        = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    metrics = {
        "model": model_name,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }

    print(f"ROC AUC: {roc_auc:.4f} | PR AUC: {pr_auc:.4f} | "
          f"Acc: {accuracy:.4f} | Prec: {precision:.4f} | Rec: {recall:.4f} | F1: {f1:.4f}")

    return model, metrics, pred


# Cell 8 – Define models & run training

In [25]:
# Cell 8 — Define classifiers and train them

# Prepare data with null-handling
train_pre = prep_nulls(train_df)
test_pre  = prep_nulls(test_df)

# Common classifier settings
rf = RandomForestClassifier(
    labelCol=LABEL_COL,
    featuresCol="features",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    numTrees=100,
    maxDepth=8,
    seed=42,
)

lr = LogisticRegression(
    labelCol=LABEL_COL,
    featuresCol="features",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=50,
    regParam=0.01,
    elasticNetParam=0.0,
)
"""

gbt = GBTClassifier(
    labelCol=LABEL_COL,
    featuresCol="features",
    predictionCol="prediction",
    maxIter=80,
    maxDepth=6,
    stepSize=0.1,
    seed=42,
)
    """

pipelines = {
    "RandomForest": Pipeline(stages=indexers + encoders + [assembler, rf]),
    "LogisticRegression": Pipeline(stages=indexers + encoders + [assembler, lr]),
   # "GBTClassifier": Pipeline(stages=indexers + encoders + [assembler, gbt]),
}

results = []
trained_models = {}

for name, pipe in pipelines.items():
    model, metrics, _ = train_and_eval(pipe, train_pre, test_pre, name)
    results.append(metrics)
    trained_models[name] = model

print("\n===== Summary of models =====")
for m in results:
    print(m)



===== Training RandomForest =====


25/12/12 15:25:57 WARN MemoryStore: Not enough space to cache rdd_1053_2 in memory! (computed 322.1 MiB so far)
25/12/12 15:25:57 WARN BlockManager: Persisting block rdd_1053_2 to disk instead.
25/12/12 15:26:38 WARN MemoryStore: Not enough space to cache rdd_1053_2 in memory! (computed 322.1 MiB so far)
25/12/12 15:26:57 WARN MemoryStore: Not enough space to cache rdd_1053_2 in memory! (computed 322.1 MiB so far)
25/12/12 15:27:16 WARN MemoryStore: Not enough space to cache rdd_1053_2 in memory! (computed 322.1 MiB so far)
25/12/12 15:27:40 WARN MemoryStore: Not enough space to cache rdd_1053_2 in memory! (computed 322.1 MiB so far)
25/12/12 15:28:13 WARN DAGScheduler: Broadcasting large task binary with size 1010.1 KiB
25/12/12 15:28:16 WARN MemoryStore: Not enough space to cache rdd_1053_2 in memory! (computed 322.1 MiB so far)
25/12/12 15:28:54 WARN DAGScheduler: Broadcasting large task binary with size 1043.7 KiB
25/12/12 15:28:57 WARN MemoryStore: Not enough space to cache rdd_10

Scoring test set with RandomForest ...


25/12/12 15:30:44 WARN DAGScheduler: Broadcasting large task binary with size 1040.8 KiB
25/12/12 15:30:51 WARN DAGScheduler: Broadcasting large task binary with size 1040.8 KiB


ROC AUC: 0.9874 | PR AUC: 0.9908 | Acc: 0.7047 | Prec: 1.0000 | Rec: 0.1893 | F1: 0.3183

===== Training LogisticRegression =====


Scoring test set with LogisticRegression ...


ROC AUC: 0.9869 | PR AUC: 0.9890 | Acc: 0.9906 | Prec: 1.0000 | Rec: 0.9742 | F1: 0.9870

===== Summary of models =====
{'model': 'RandomForest', 'roc_auc': 0.987404471063458, 'pr_auc': 0.9907509480060281, 'accuracy': 0.7047248659786458, 'precision': 1.0, 'recall': 0.18930766781769648, 'f1': 0.3183493606243436, 'tp': 7730, 'tn': 71276, 'fp': 0, 'fn': 33103}
{'model': 'LogisticRegression', 'roc_auc': 0.9869240167278698, 'pr_auc': 0.988978541033193, 'accuracy': 0.9906162752321401, 'precision': 1.0, 'recall': 0.9742365243797908, 'f1': 0.9869501575408738, 'tp': 39781, 'tn': 71276, 'fp': 0, 'fn': 1052}


# Cell 9 – Choose best model & save to ADLS

In [26]:
# Cell 9 — Pick best model (by ROC AUC) and save it

# Pick the model with the highest ROC AUC
best = sorted(results, key=lambda r: r["roc_auc"], reverse=True)[0]
best_name = best["model"]
best_model = trained_models[best_name]

print(f"\nBest model by ROC AUC: {best_name} (ROC AUC={best['roc_auc']:.4f})")

# Save the whole PipelineModel to ADLS so scoring can reuse preprocessing + classifier
best_model.write().overwrite().save(MODEL_PATH)
print(f"✅ Saved best model to: {MODEL_PATH}")



Best model by ROC AUC: RandomForest (ROC AUC=0.9874)


✅ Saved best model to: abfss://golddata@clientdatastorage.dfs.core.windows.net/models/claims_risk_best


# Cell 10 – Load model & score sample claims

In [27]:
# Cell 10 — Load persisted model and score sample claims

loaded_model = PipelineModel.load(MODEL_PATH)

# Take a random sample of test claims
sample_to_score = (
    test_df
        .orderBy(F.rand())
        .limit(20)
)

scored_raw = loaded_model.transform(prep_nulls(sample_to_score))

# Convert probability vector to array and take probability of class 1 (fraud)
scored = (
    scored_raw
        .withColumn("prob_arr", vector_to_array("probability"))
        .withColumn("fraud_prob", F.col("prob_arr")[1])
        .select(
            *[c for c in id_cols if c in scored_raw.columns],
            LABEL_COL,
            "fraud_prob",
            "prediction",
            "Claim_Type_Name",
            "Claim_Status",
            "Claim_Amount_GBP",
            "Payout_GBP",
            "Payout_to_Amount_Ratio",
            "Provider_Risk_Tier",
        )
)

scored.printSchema()
scored.show(truncate=False)


root
 |-- Claim_ID: string (nullable = true)
 |-- Member_Key: string (nullable = true)
 |-- Provider_ID: string (nullable = true)
 |-- Is_Fraudulent_Claim: double (nullable = true)
 |-- fraud_prob: double (nullable = true)
 |-- prediction: double (nullable = false)
 |-- Claim_Type_Name: string (nullable = true)
 |-- Claim_Status: string (nullable = true)
 |-- Claim_Amount_GBP: double (nullable = true)
 |-- Payout_GBP: double (nullable = true)
 |-- Payout_to_Amount_Ratio: double (nullable = true)
 |-- Provider_Risk_Tier: string (nullable = true)



+---------+----------+-----------+-------------------+-------------------+----------+---------------+------------+------------------+----------+----------------------+------------------+
|Claim_ID |Member_Key|Provider_ID|Is_Fraudulent_Claim|fraud_prob         |prediction|Claim_Type_Name|Claim_Status|Claim_Amount_GBP  |Payout_GBP|Payout_to_Amount_Ratio|Provider_Risk_Tier|
+---------+----------+-----------+-------------------+-------------------+----------+---------------+------------+------------------+----------+----------------------+------------------+
|CLM701704|BENE45984 |PRV55208   |0.0                |0.30197347924196893|0.0       |Outpatient     |Settled     |11.594380984233357|10.0      |0.8624867522982487    |Low risk          |
|CLM380189|BENE103641|PRV54121   |0.0                |0.30197347924196893|0.0       |Outpatient     |Pending     |137.4312491737245 |100.0     |0.7276365499202565    |Low risk          |
|CLM428553|BENE98227 |PRV56583   |0.0                |0.301973479

# 📌 Cell 11 — MLflow Training Wrapper (FRAUD MODEL)

In [36]:
# Cell 11 — Log full MLflow run with params, metrics, and model artifact

import mlflow
import mlflow.spark
from datetime import datetime

with mlflow.start_run(run_name="fraud_detection_training"):

    # -----------------------------
    # 🔹 Log dataset info
    # -----------------------------
    mlflow.log_param("train_rows", train_df.count())
    mlflow.log_param("test_rows", test_df.count())

    # -----------------------------
    # 🔹 Log features & label
    # -----------------------------
    mlflow.log_param("label_col", LABEL_COL)
    mlflow.log_param("numeric_features", ",".join(numeric_cols))
    mlflow.log_param("categorical_features", ",".join(cat_cols))

    # -----------------------------
    # 🔹 Log metrics for each model
    # -----------------------------
    for m in results:
        model_name = m["model"]
        prefix = model_name.replace(" ", "_")

        mlflow.log_metric(f"{prefix}_auc_roc", m["roc_auc"])
        mlflow.log_metric(f"{prefix}_auc_pr",  m["pr_auc"])
        mlflow.log_metric(f"{prefix}_accuracy", m["accuracy"])
        mlflow.log_metric(f"{prefix}_precision", m["precision"])
        mlflow.log_metric(f"{prefix}_recall", m["recall"])
        mlflow.log_metric(f"{prefix}_f1", m["f1"])

        # confusion matrix
        mlflow.log_metric(f"{prefix}_tp", m["tp"])
        mlflow.log_metric(f"{prefix}_tn", m["tn"])
        mlflow.log_metric(f"{prefix}_fp", m["fp"])
        mlflow.log_metric(f"{prefix}_fn", m["fn"])

    # -----------------------------
    # 🔹 Log winning model metadata
    # -----------------------------
    mlflow.log_param("best_model", best_name)

    # -----------------------------
    # 🔹 Log model hyperparameters
    # -----------------------------
    if best_name == "RandomForest":
        mlflow.log_param("numTrees", rf.getNumTrees())
        mlflow.log_param("maxDepth", rf.getOrDefault("maxDepth"))
        mlflow.log_param("algo", "RandomForest")

    elif best_name == "LogisticRegression":
        mlflow.log_param("maxIter", lr.getOrDefault("maxIter"))
        mlflow.log_param("regParam", lr.getOrDefault("regParam"))
        mlflow.log_param("algo", "LogisticRegression")

    # -----------------------------
    # 🔹 Log the actual Spark model
    # -----------------------------
    mlflow.spark.log_model(
        spark_model=best_model,
        artifact_path="fraud_best_model",
        registered_model_name=None  # you can register later manually
    )

    print("✅ Logged MLflow run to experiment 'bupa_claims_fraud'")
print("👉 You can open MLflow UI with:  mlflow ui --backend-store-uri", TRACKING_URI)


✅ Logged MLflow run to experiment 'bupa_claims_fraud'
👉 You can open MLflow UI with:  mlflow ui --backend-store-uri file:/Users/manojrammopati/Public/Projects/bupa_insurance_project/mlruns


# Cell 12 Model Registration in MLFLOW

from datetime import datetime
import mlflow
import mlflow.spark
from mlflow.tracking import MlflowClient

client = MlflowClient()

with mlflow.start_run(run_name="fraud_detection_training") as run:

    ###### ... all your logging from before ...

    ###### log the best model
    mlflow.spark.log_model(
        spark_model=best_model,
        artifact_path="fraud_best_model",
        registered_model_name=None,
    )

    ###### 👇 capture run_id while run is still active
    run_id = run.info.run_id

print("✅ MLflow logging completed for fraud model.")

###### OPTIONAL: register after the run closes
model_uri = f"runs:/{run_id}/fraud_best_model"

###### This will create the registry entry the first time; if it already exists, comment this out
try:
    client.create_registered_model("bupa_fraud_model")
except Exception as e:
    # usually "RESOURCE_ALREADY_EXISTS" after first time – safe to ignore
    print("Registered model may already exist:", e)

client.create_model_version(
    name="bupa_fraud_model",
    source=model_uri,
)

print("📌 Registered new model version for bupa_fraud_model from run:", run_id)


# 📈 Cell 13 SAVING ALL MODELS METRICS TO ML_MONITORING

In [34]:
from datetime import datetime
import utils_silver

paths_bronze, paths_silver, paths_gold = utils_silver.build_paths(ACCOUNT)

now_ts = datetime.utcnow()
rows = []

for m in results:
    rows.append({
        "model_name":   m["model"],
        "use_case":     "claims_fraud",
        "dataset_name": "ft_claims_risk",
        "dataset_split": "test",
        "auc":       float(m.get("roc_auc", m.get("auc_roc", 0.0))),
        "accuracy":  float(m.get("accuracy", 0.0)),
        "precision": float(m.get("precision", 0.0)),
        "recall":    float(m.get("recall", 0.0)),
        "f1":        float(m.get("f1", 0.0)),
        "ts":        now_ts,
        "notes":     "fraud model sweep",
    })

utils_silver.write_ml_monitoring(
    spark=spark,
    rows=rows,
    paths_gold=paths_gold,
)


[ML_MON] Wrote 2 rows to abfss://golddata@clientdatastorage.dfs.core.windows.net/ml_monitoring


In [30]:
df = spark.read.format("delta").load(
    "abfss://golddata@clientdatastorage.dfs.core.windows.net/ml_monitoring"
)

df.printSchema()
df.show()

root
 |-- run_ts: timestamp (nullable = true)
 |-- model_name: string (nullable = true)
 |-- use_case: string (nullable = true)
 |-- dataset_name: string (nullable = true)
 |-- dataset_split: string (nullable = true)
 |-- auc: double (nullable = true)
 |-- accuracy: double (nullable = true)
 |-- precision: double (nullable = true)
 |-- recall: double (nullable = true)
 |-- f1: double (nullable = true)
 |-- notes: string (nullable = true)



+--------------------+--------------------+--------------------+---------------+-------------+------------------+------------------+------------------+-------------------+------------------+--------------------+
|              run_ts|          model_name|            use_case|   dataset_name|dataset_split|               auc|          accuracy|         precision|             recall|                f1|               notes|
+--------------------+--------------------+--------------------+---------------+-------------+------------------+------------------+------------------+-------------------+------------------+--------------------+
|2025-12-11 14:38:...|BestPolicyChurnModel|policy_churn_scoring|ft_policy_churn|        score|               0.0|               0.0|               0.0|                0.0|               0.0|Batch scoring run...|
|2025-12-10 21:02:...|        RandomForest|        claims_fraud| ft_claims_risk|         test| 0.987404471063458|0.7047248659786458|               1.0|0